## Imports

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [6]:
from pyxlsb import open_workbook

In [7]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [8]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [9]:
# Load data from 'Elite Customer data.xlsb'
file_path = "Elite Customer data.xlsb"  # Ensure this file is in your current directory

# Read a specific sheet (replace 'Sheet1' with the actual sheet name)
df = pd.read_excel(file_path, sheet_name='Sheet1', engine='pyxlsb')

## Datasets & Parameter

In [10]:
df

,city,gender,user_id,customer_obj_mobile,taxi_retention_segments,Customer_Obj_Name
0,Delhi,female,625d1422b9eaf550004a3b6f,8178430135,ELITE,Pooja
1,Delhi,male,5daa690a52047954050e05e9,9667116049,ELITE,Sunny Vats
2,Hyderabad,male,64b98f0f218ffc447c9ad73a,9849410041,ELITE,Parthiban Abhiramacheri
3,Bangalore,male,642086d1c541dacb4f19f7f5,9845720987,ELITE,Vijaykumar S
4,Bangalore,male,602b3a2319e49823b91f6641,8130507623,ELITE,Vishesh Verma
...,...,...,...,...,...,...
824,Mumbai,female,63b1293aa9f27531da9eed81,9665725753,ELITE,Sunita Nair
825,Mumbai,female,65dd5ab9fc33e16621c4e223,7887753543,ELITE,Chandni Khatri
826,Mumbai,female,625d9ed3797562256b3bed6b,9930790789,ELITE,Yukta Borkar
827,Mumbai,male,67623e714a965b2ba8d40e2c,9819861820,ELITE,Amey


In [30]:
df.user_id.nunique()

812

In [12]:
customer_list = df['user_id'].to_list()
len(customer_list)

829

In [29]:
829 - 842

-13

In [15]:
customer_str = ", ".join(f"'{customer}'" for customer in customer_list)
print(customer_str)

'625d1422b9eaf550004a3b6f', '5daa690a52047954050e05e9', '64b98f0f218ffc447c9ad73a', '642086d1c541dacb4f19f7f5', '602b3a2319e49823b91f6641', '661d07f69c553250f4663dac', '65cab9d06cc0ecbf0e4ae24a', '66c5e5e87e612e28343ac5bd', '63d366cdf17fa764ceedf141', '5cf4a712c0b26018f5d7210e', '62db6a8fcb61570f63578a18', '65f854ba0087bd4d5f4afd79', '629b5e02a9c7b43889fbb098', '630da253744a5c204181e8dd', '641d25a4a4eb4702b8ab85f0', '5fc9a6c45441f851e343a732', '5d77b94bd0286d106d84539e', '61b5e04a61bb51102dcbcbf5', '5cc2b3713d65ca5e2563617a', '62c230ba17af9a77be69731d', '5e46851e981e4336e2e20f22', '63ecd808d6d9b2097fe87c5a', '627b59f4c75b904f52072a1b', '656ef8f7100f1b7ab43c0dc5', '6702e9a53d68e97659d6737f', '5cf4a712c0b26018f5d7210e', '5d56259455fbf50d45f72f27', '675a78e75bff40bffb47d1df', '65b61a23157cf55fe4e60d9e', '60f6febb8f3479d97d1adebe', '63fe44288e212c64f44ab363', '6115012b8dddcb578c6d0eaf', '6641fc1c54d49f2b96e7c248', '65f698117ca1e4915fad39b7', '661415a2c142c14d90144cd2', '62456e2e5fd0224ece6

In [22]:
## base

base_query = f"""

with csv as (
    select 
        customerId,
        mobileOS,
        deviceManufacturer
    from 
        datasets.customer_single_view
    where 
        customerId IN ({customer_str})
    ),
    
    appo as (
    select 
        userid,
        case when regexp_like(lower(appographydata), 'zomato') then '1' else '0' end zomato,
        case when regexp_like(lower(appographydata), 'swiggy') then '1' else '0' end swiggy
    from 
        datasets.customer_apps_installed_snapshot
    where 
        userid IN ({customer_str})
    )
    
    select
        coalesce(customerId, userid) customer_id,
        mobileOS,
        deviceManufacturer,
        coalesce(zomato, 'UNKNOWN') zomato,
        coalesce(swiggy, 'UNKNOWN') swiggy
    from 
        csv a
    full join 
        appo b
        on customerId = userid

"""

df_base_query = pd.read_sql(base_query, connection)
df_base_query

,customer_id,mobileOS,deviceManufacturer,zomato,swiggy
0,6303bb0104cb1b28700dcc78,ANDROID,vivo,0,0
1,62dea142c389d9458f90d96c,ANDROID,samsung,1,1
2,67602d6617b102345dfc67b4,ANDROID,ITEL,1,0
3,5ccbb04840e03a5c70b88678,ANDROID,OnePlus,0,0
4,6774656166024a84214990aa,ANDROID,realme,0,0
...,...,...,...,...,...
837,61a653c417dd6abc32060d74,ANDROID,TECNO MOBILE LIMITED,1,1
838,639df45ccd56c04285838951,ANDROID,OnePlus,1,1
839,663646ae051e29e3287b83de,ANDROID,samsung,1,1
840,672f5d96bb419ab48b0368e5,ANDROID,motorola,1,0


In [31]:
df_base_query.customer_id.nunique()

812

In [23]:
df_base_query.shape

(842, 5)

In [24]:
df_base_query[ df_base_query['mobileOS'] == 'IOS']

,customer_id,mobileOS,deviceManufacturer,zomato,swiggy
33,652f177d1e930b6ca2544107,IOS,Apple,UNKNOWN,UNKNOWN
91,6299fbbb734ac520cd0bbeec,IOS,Apple,UNKNOWN,UNKNOWN
118,6618a8fe05d4ee1af982e9d1,IOS,Apple,1,1
267,66be0adc0eec199535d53c14,IOS,Apple,1,1
274,662f6b42396311c68a94212f,IOS,Apple,1,1
285,61854194692d9de8917cb9da,IOS,Apple,UNKNOWN,UNKNOWN
308,6115012b8dddcb578c6d0eaf,IOS,Apple,1,0
361,676a3bcc91c9d1930d54a4ba,IOS,Apple,UNKNOWN,UNKNOWN
362,62addccb214ba0106df329ad,IOS,Apple,1,1
402,629c6a8da3627ea2578d9351,IOS,Apple,UNKNOWN,UNKNOWN


In [26]:
df_final = pd.merge(df, df_base_query[['customer_id', 'zomato', 'swiggy']], how='left', left_on = 'user_id', right_on='customer_id')
df_final

,city,gender,user_id,customer_obj_mobile,taxi_retention_segments,Customer_Obj_Name,customer_id,zomato,swiggy
0,Delhi,female,625d1422b9eaf550004a3b6f,8178430135,ELITE,Pooja,625d1422b9eaf550004a3b6f,0,0
1,Delhi,male,5daa690a52047954050e05e9,9667116049,ELITE,Sunny Vats,5daa690a52047954050e05e9,0,0
2,Hyderabad,male,64b98f0f218ffc447c9ad73a,9849410041,ELITE,Parthiban Abhiramacheri,64b98f0f218ffc447c9ad73a,0,0
3,Bangalore,male,642086d1c541dacb4f19f7f5,9845720987,ELITE,Vijaykumar S,642086d1c541dacb4f19f7f5,0,0
4,Bangalore,male,602b3a2319e49823b91f6641,8130507623,ELITE,Vishesh Verma,602b3a2319e49823b91f6641,1,1
...,...,...,...,...,...,...,...,...,...
854,Mumbai,female,63b1293aa9f27531da9eed81,9665725753,ELITE,Sunita Nair,63b1293aa9f27531da9eed81,1,0
855,Mumbai,female,65dd5ab9fc33e16621c4e223,7887753543,ELITE,Chandni Khatri,65dd5ab9fc33e16621c4e223,1,1
856,Mumbai,female,625d9ed3797562256b3bed6b,9930790789,ELITE,Yukta Borkar,625d9ed3797562256b3bed6b,1,0
857,Mumbai,male,67623e714a965b2ba8d40e2c,9819861820,ELITE,Amey,67623e714a965b2ba8d40e2c,1,1


In [28]:
df_final.to_clipboard(index=False)

In [32]:
df_final_v1 = df_final.drop_duplicates(keep='last')
df_final_v1

,city,gender,user_id,customer_obj_mobile,taxi_retention_segments,Customer_Obj_Name,customer_id,zomato,swiggy
0,Delhi,female,625d1422b9eaf550004a3b6f,8178430135,ELITE,Pooja,625d1422b9eaf550004a3b6f,0,0
1,Delhi,male,5daa690a52047954050e05e9,9667116049,ELITE,Sunny Vats,5daa690a52047954050e05e9,0,0
2,Hyderabad,male,64b98f0f218ffc447c9ad73a,9849410041,ELITE,Parthiban Abhiramacheri,64b98f0f218ffc447c9ad73a,0,0
3,Bangalore,male,642086d1c541dacb4f19f7f5,9845720987,ELITE,Vijaykumar S,642086d1c541dacb4f19f7f5,0,0
4,Bangalore,male,602b3a2319e49823b91f6641,8130507623,ELITE,Vishesh Verma,602b3a2319e49823b91f6641,1,1
...,...,...,...,...,...,...,...,...,...
854,Mumbai,female,63b1293aa9f27531da9eed81,9665725753,ELITE,Sunita Nair,63b1293aa9f27531da9eed81,1,0
855,Mumbai,female,65dd5ab9fc33e16621c4e223,7887753543,ELITE,Chandni Khatri,65dd5ab9fc33e16621c4e223,1,1
856,Mumbai,female,625d9ed3797562256b3bed6b,9930790789,ELITE,Yukta Borkar,625d9ed3797562256b3bed6b,1,0
857,Mumbai,male,67623e714a965b2ba8d40e2c,9819861820,ELITE,Amey,67623e714a965b2ba8d40e2c,1,1


In [33]:
df_final_v1.to_clipboard(index=False)

In [36]:
df_final_v1[ df_final_v1['zomato'] == 'UNKNOWN'].user_id.nunique()

19

In [11]:
df_base_query.to_parquet('data_dump_for_process_mining_bangalore.parquet', index=False)

In [12]:
test1 = pd.read_parquet('data_dump_for_process_mining_without_payment.parquet')

In [13]:
test1.shape

(1338288, 9)

In [15]:
1341310-1338288

3022

In [18]:
## base

base_query = f"""

SELECT
    city AS city,
    pickup_cluster,
    user_id as customer_id,
    fare_estimate_id,service_detail_id as service_id_fare,
    api_context,
    date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
    cast(final_amount AS double) AS fe_amount,polyline,
    epoch
FROM
    hive.pricing.fare_estimates_enriched
WHERE
     yyyymmdd = '20241021'             
     and service_name in ('Link')
     AND city = 'Bangalore'
     AND api_context='/fare/estimate'
"""

df_base_query = pd.read_sql(base_query, connection)
df_base_query

DatabaseError: {'message': 'Query exceeded the maximum execution time limit of 10.00m', 'errorCode': 131075, 'errorName': 'EXCEEDED_TIME_LIMIT', 'errorType': 'INSUFFICIENT_RESOURCES', 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': 'Query exceeded the maximum execution time limit of 10.00m', 'suppressed': [], 'stack': ['io.trino.execution.QueryTracker.enforceTimeLimits(QueryTracker.java:187)', 'io.trino.execution.QueryTracker.lambda$start$0(QueryTracker.java:89)', 'com.google.common.util.concurrent.MoreExecutors$ScheduledListeningDecorator$NeverSuccessfulListenableFutureTask.run(MoreExecutors.java:730)', 'java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)', 'java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)', 'java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)', 'java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.jav

In [ ]:
df_base_query.to_parquet('fare_estimates_bangalore_link_20241021.parquet', index=False)